In [ ]:
import os
import torch
import numpy as np
import pickle
from sklearn.decomposition import PCA
from tabular_model import *
from tabular_util import *

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True

_, config = load_config_yaml("config.yaml")
config["device"] = torch.device("cpu")  # ('cuda:'+ config['gpu'])

In [ ]:
def infer(model, infer_csv_path, output_name, pca):
    model.eval()

    infer_df = pd.read_csv(infer_csv_path)
    infer_dataset = LongitudinalSingleDataset(infer_df)
    inferDataLoader = DataLoader(
        infer_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=0,
    )

    result_dir = f'./results/{config["dataset_name"]}/{config["model_name"]}/{ckpt_label}/'
    os.makedirs(result_dir, exist_ok=True)
    path = os.path.join(result_dir, output_name if output_name.endswith(".npy") else output_name + ".npy")

    rid_list, tab_list, lb_list, recon_list, z_list, age_list = [], [], [], [], [], []

    with torch.no_grad():
        for _, sample in enumerate(inferDataLoader, 0):
            tab = sample["tab"].to(config["device"], dtype=torch.float).unsqueeze(1)
            if not torch.isfinite(tab).all():
                nan_indices = torch.nonzero(torch.isnan(tab))
                for i, _, j in nan_indices:
                    tab[i, _, j] = torch.nanmean(tab[:, :, j])
            zero_mx = torch.zeros_like(tab)
            zs, recons = model(tab, zero_mx)

            rid_list.extend(sample["rid"])
            tab_list.append(tab.detach().cpu().numpy())
            recon_list.append(recons[0].detach().cpu().numpy())
            z_list.append(zs[0].detach().cpu().numpy())
            age_list.append(sample["age"].numpy())
            lb_list.append(sample["lb"].detach().cpu().numpy())

    # Concatenate arrays
    tab_list = np.concatenate(tab_list, axis=0)
    recon_list = np.concatenate(recon_list, axis=0)
    z_list = np.concatenate(z_list, axis=0)
    age_list = np.concatenate(age_list, axis=0)
    lb_list = np.concatenate(lb_list, axis=0)

    # PCA processing
    if pca is None:
        pca_model = PCA(n_components=10)
        pcs = pca_model.fit_transform(z_list)
        pickle_path = os.path.join(result_dir, "pca10_transformer.pkl")
        with open(pickle_path, "wb") as f:
            pickle.dump(pca_model, f)
        print(f"Fitted new PCA and saved transformer to {pickle_path}")

    # Save results
    results = {
        "RID": np.array(rid_list),
        "age": age_list,
        "lb": lb_list,
        "tab": tab_list,
        "recon": recon_list,
        "z": z_list,
        "pc": pcs,  # shape (n_samples, 2)
    }
    # np.save(path, results, allow_pickle=True)
    # print(f"Saved inference results to {path}")
    return results


def load_ckpt(ckpt_path):
    flag, config_load = load_config_yaml(os.path.join(ckpt_path, "config.yaml"))
    # load config file
    if flag:
        print("load yaml config file")
        for key in config_load.keys():  # if yaml has, use yaml's param, else use config
            if key == "phase" or key == "gpu" or key == "continue_train" or key == "ckpt_name":
                continue
            if key in config.keys():
                config[key] = config_load[key]
            else:
                print("current config do not have yaml param")
    else:
        save_config_yaml(ckpt_path, config)

    # Load model
    model = LSP(
        num_neighbours=config["num_neighbours"],
        dims=config["dims"],
        agg_method=config["agg_method"],
        gpu=config["device"],
        activation="leakyrelu",
        dropout=0.0,
        slope=0.2,
        batch_norm=True,
    ).to(config["device"])

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=1e-5, amsgrad=True)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5, min_lr=1e-5)

    [optimizer, scheduler, model], start_epoch = load_checkpoint_by_key(
        [optimizer, scheduler, model],
        ckpt_path,
        ["optimizer", "scheduler", "model"],
        config["device"],
        config["ckpt_name"],
    )
    print(model.parameters())

    return config, model, optimizer, scheduler, start_epoch


######################
# Inference settings #
######################

trial = 2
infer_name = "train"
ckpt_dir = "./ckpt/ADNI1GO234/LSP/"
ckpt_label = next(folder for folder in os.listdir(ckpt_dir) if folder.startswith(f"trial{trial}_"))
config["ckpt_path"] = os.path.join("./ckpt/", config["dataset_name"], config["model_name"], ckpt_label)

config, model, optimizer, scheduler, start_epoch = load_ckpt(config["ckpt_path"])

infer_csv_path = f"data/ADNI1GO234/splits/trial{trial}/preadj_{infer_name}.csv"

output_name = f"{infer_name}_results.npy"
# pca = os.path.join("./results/", config["dataset_name"], config["model_name"], ckpt_label, "pca_transformer.pkl")
print(f"\nInference for {infer_name} data, trial {trial}...")
results = infer(model, infer_csv_path, output_name, pca=None)

In [ ]:
pca_path = "results/ADNI1GO234/LSP/trial2_2025_9_24_15_24/pca10_transformer.pkl"
with open(pca_path, "rb") as f:
    pca_model = pickle.load(f)
    
# print pca model eigenvalues and explained variance ratio
print("PCA Components (Eigenvalues):", pca_model.explained_variance_)
print("Explained Variance Ratio:", pca_model.explained_variance_ratio_)

# Scree plot
import matplotlib.pyplot as plt
explained_variance = pca_model.explained_variance_
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')
plt.xticks(range(1, len(explained_variance) + 1))
plt.grid()
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

pc_data = results["pc"]
legend = results["lb"].flatten()

# Use the first three principal components for 3D visualization.
df = pd.DataFrame(
    pc_data[:, [0, 1, 2]],
    columns=["PC1", "PC2", "PC3"]
)
df["Legend"] = legend

# highligh only target value, others grey
target_value = 4   # Change the highlighted value here.

df["Color"] = df["Legend"].apply(lambda x: x if x == target_value else "Other")

# Interactive 3D scatter plot.
fig = px.scatter_3d(
    df,
    x="PC1", y="PC2", z="PC3",
    color="Color",
    color_discrete_map={
        target_value: "red",   # Highlight color.
        "Other": "lightgray"   # Color for all other values.
    },
    opacity=0.8,
    title=f"Highlight Legend = {target_value}",
    hover_data=["Legend", "PC1", "PC2", "PC3"]
)

# marker size
fig.update_traces(marker=dict(size=4))

# Layout
fig.update_layout(
    width=700,
    height=600,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()


In [ ]:
import pandas as pd
import plotly.express as px

pc_data = results["pc"]
legend = results["age"].flatten()

# Use the first three principal components for 3D visualization.
df = pd.DataFrame(
    pc_data[:, [0, 1, 2]],
    columns=["PC1", "PC2", "PC3"]
)
df["Legend"] = legend

# Exclude rows where Legend equals -1.
df = df[df["Legend"] != -1]

# Interactive 3D scatter plot.
fig = px.scatter_3d(
    df,
    x="PC1", y="PC2", z="PC3",
    color="Legend",
    opacity=0.8,
    title="PCA 3D Visualization",
    hover_data=["Legend", "PC1", "PC2", "PC3"]
)

# marker size
fig.update_traces(marker=dict(size=4))

# Layout
fig.update_layout(
    width=700,
    height=600,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()

In [ ]:
# Plot a scatter plot matrix of the first 4 principal components colored by legend
import plotly.express as px
pc_data = results["pc"]
legend = results["lb"].flatten()
df = pd.DataFrame(
    pc_data[:, :4],
    columns=["PC1", "PC2", "PC3", "PC4"]
)
df["Legend"] = legend
fig = px.scatter_matrix(
    df,
    dimensions=["PC1", "PC2", "PC3", "PC4"],
    color="Legend",
    title="Scatter Matrix of First 4 Principal Components Colored by Legend",
    opacity=0.7,
    hover_data=["Legend"]
)
fig.update_traces(diagonal_visible=False)
fig.update_layout(
    width=800,
    height=800,
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()